# Notebook 03: Feature Engineering

**Project 4 -- End-to-End Machine Learning: LendingClub Loan Default Prediction**

This notebook transforms the cleaned dataset into an intermediate feature set. **Data-dependent transforms** (scaling, target encoding, NZV/correlation filtering) are deferred to Notebook 04, where they are fit on training data only to prevent information leakage.

Steps performed here (deterministic, no data-dependent statistics):
1. Separate temporal split index (`issue_d`)
2. Merge FICO into single feature
3. Create derived ratio features
4. Log-transform skewed features
5. Add missingness indicators for sentinel-filled columns
6. Consolidate rare categories (home_ownership, purpose)
7. Ordinal encode sub_grade (domain-driven A1=1...G5=35)
8. One-hot encode low-cardinality categoricals
9. Keep `addr_state` as string (target-encoded in NB04 on train only)

**Deferred to Notebook 04 (fit on training data only):**
- StandardScaler
- addr_state target encoding
- Near-zero-variance filtering
- High-correlation feature selection

**Input:** `data/cleaned.parquet`
**Output:** `data/engineered.parquet`, `data/temporal_split_index.parquet`

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

print("Libraries loaded.")

## 1. Load & Separate Temporal Split Index

In [ ]:
df = pd.read_parquet('data/cleaned.parquet')
print(f"Loaded: {df.shape}")

# Save issue_d for temporal train/test split in Notebook 04
temporal_idx = df[['issue_d']].copy()
temporal_idx.to_parquet('data/temporal_split_index.parquet', index=True)
print(f"Saved temporal_split_index.parquet — range: {df['issue_d'].min()} to {df['issue_d'].max()}")

# Drop issue_d from feature set
df = df.drop(columns=['issue_d'])
print(f"Shape after dropping issue_d: {df.shape}")

## 2. Merge FICO into Single Feature

`fico_range_low` and `fico_range_high` are always exactly 4 apart. Merge into a single `fico_avg` to reduce redundancy.

In [ ]:
# Verify they're always 4 apart
diff = df['fico_range_high'] - df['fico_range_low']
print(f"FICO range difference — unique values: {diff.unique()}")

df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
df = df.drop(columns=['fico_range_low', 'fico_range_high'])
print(f"Created fico_avg — range: [{df['fico_avg'].min()}, {df['fico_avg'].max()}]")
print(f"Shape: {df.shape}")

## 3. Create Derived Ratio Features

Domain-driven features capturing financial stress signals:

In [ ]:
# Avoid division by zero with small epsilon
eps = 1.0

# Monthly installment as % of monthly income
df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12 + eps)

# Loan amount relative to annual income
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + eps)

# Revolving balance relative to annual income
df['revol_bal_to_income'] = df['revol_bal'] / (df['annual_inc'] + eps)

# Proportion of accounts currently open
df['open_acc_ratio'] = df['open_acc'] / (df['total_acc'] + eps)

new_features = ['installment_to_income', 'loan_to_income', 'revol_bal_to_income', 'open_acc_ratio']
print("New ratio features:")
print(df[new_features].describe().round(4).to_string())
print(f"\nShape: {df.shape}")

## 4. Log-Transform Skewed Features

`annual_inc`, `revol_bal`, and `tot_cur_bal` have extreme right skew. Adding log-transformed versions improves model performance on linear models.

In [ ]:
log_cols = ['annual_inc', 'revol_bal', 'tot_cur_bal']

for col in log_cols:
    new_col = f'log_{col}'
    df[new_col] = np.log1p(df[col])
    print(f"  {new_col}: range [{df[new_col].min():.2f}, {df[new_col].max():.2f}]")

print(f"\nShape: {df.shape}")

## 5. Encode Categorical Variables

| Feature | Method | Rationale |
|---------|--------|-----------|
| `grade` | Drop (keep sub_grade) | sub_grade is more granular |
| `sub_grade` | Ordinal (A1=1...G5=35) | Natural hierarchy (domain mapping, not data-derived) |
| `home_ownership` | Consolidate rare + one-hot | Low cardinality; ANY/NONE/OTHER -> OTHER |
| `purpose` | Consolidate rare + one-hot | educational/renewable_energy/wedding -> other |
| `verification_status` | One-hot | 3 levels |
| `application_type` | One-hot | 2 levels |
| `initial_list_status` | One-hot | 2 levels |
| `addr_state` | **Kept as string** | Target-encoded in NB04 on training data only |

In [ ]:
# --- sub_grade: ordinal encode A1=1 ... G5=35 ---
subgrade_order = []
for letter in 'ABCDEFG':
    for num in range(1, 6):
        subgrade_order.append(f'{letter}{num}')

subgrade_map = {sg: i + 1 for i, sg in enumerate(subgrade_order)}
df['sub_grade'] = df['sub_grade'].map(subgrade_map)
print(f"sub_grade encoded: range [{df['sub_grade'].min()}, {df['sub_grade'].max()}]")

# Drop grade (sub_grade captures the same info at finer granularity)
df = df.drop(columns=['grade'])
print("Dropped grade (keeping sub_grade)")

In [ ]:
# --- home_ownership: consolidate rare categories ---
print(f"home_ownership before: {df['home_ownership'].value_counts().to_dict()}")
rare_home = ['ANY', 'NONE', 'OTHER']
df['home_ownership'] = df['home_ownership'].replace(rare_home, 'OTHER')
print(f"home_ownership after:  {df['home_ownership'].value_counts().to_dict()}")

# --- purpose: consolidate rare categories BEFORE one-hot encoding ---
# These would become >99.5% zero after one-hot and get silently NZV-removed.
# Explicit consolidation is cleaner and preserves the signal in 'other'.
print(f"\npurpose before: {df['purpose'].value_counts().to_dict()}")
rare_purposes = ['educational', 'renewable_energy', 'wedding']
df['purpose'] = df['purpose'].replace(rare_purposes, 'other')
print(f"purpose after:  {df['purpose'].nunique()} categories")
print(df['purpose'].value_counts().to_string())

## 4.5 Add Missingness Indicators

Notebook 01 used sentinel fills for `mths_since_*` columns where NaN means "event never happened" (e.g., no delinquency). The sentinel value (column max + 1) encodes meaningful structural missingness. We add explicit binary flags so models can learn from this signal directly.

In [ ]:
# Reconstruct missingness indicators from NB01's sentinel fills
# NB01 filled NaN with max+1 for "event never happened" columns
sentinel_map = {
    'mths_since_last_delinq': 227,       # max was 226, NaN = never delinquent
    'mths_since_recent_bc_dlq': 203,     # max was 202, NaN = no bankcard delinquency
    'mths_since_recent_revol_delinq': 203, # max was 202, NaN = no revolving delinquency
    'mths_since_recent_inq': 26,         # max was 25, NaN = no recent inquiry
}

print("Missingness indicators (sentinel-based):")
for col, sentinel_val in sentinel_map.items():
    flag_col = f'{col}_missing'
    df[flag_col] = (df[col] >= sentinel_val).astype(int)
    pct = df[flag_col].mean() * 100
    print(f"  {flag_col}: {df[flag_col].sum():,} rows ({pct:.1f}%)")

print(f"\nShape after adding missingness indicators: {df.shape}")

In [ ]:
# --- One-hot encode low-cardinality categoricals ---
# addr_state is intentionally LEFT as a string column.
# It will be target-encoded in NB04 after the temporal split (train labels only).
onehot_cols = ['home_ownership', 'purpose', 'verification_status',
               'application_type', 'initial_list_status']

print("One-hot encoding:")
for col in onehot_cols:
    n_unique = df[col].nunique()
    print(f"  {col}: {n_unique} categories")

df = pd.get_dummies(df, columns=onehot_cols, drop_first=True, dtype=int)

print(f"\naddr_state: kept as string ({df['addr_state'].nunique()} unique states)")
print(f"Shape after one-hot encoding: {df.shape}")

## 6. Save & Summary

Output `engineered.parquet` -- an intermediate feature set with:
- Unscaled numeric features
- `addr_state` as string (target-encoded in NB04)
- No NZV or correlation filtering (done in NB04 on train only)
- Missingness indicators for sentinel-filled columns

In [ ]:
df.to_parquet('data/engineered.parquet', index=True)

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"Final shape: {df.shape}")
print(f"Target distribution:")
print(df['loan_status'].value_counts())
print(f"\nDefault rate: {df['loan_status'].mean():.2%}")

feature_cols = [c for c in df.columns if c != 'loan_status']
numeric_cols = [c for c in feature_cols if df[c].dtype in ['float64', 'int64', 'Int64'] and c != 'addr_state']
string_cols = [c for c in feature_cols if df[c].dtype == 'object']

print(f"\nFeature breakdown:")
print(f"  Numeric (unscaled): {len(numeric_cols)}")
print(f"  String (addr_state, to be target-encoded in NB04): {len(string_cols)}")
print(f"\nDeferred to NB04 (fit on training data only):")
print(f"  - StandardScaler")
print(f"  - addr_state target encoding")
print(f"  - Near-zero-variance filter")
print(f"  - High-correlation filter")
print(f"\nAll features ({len(feature_cols)}):")
print(feature_cols)

In [ ]:
# Quick verification: reload and check
df_check = pd.read_parquet('data/engineered.parquet')
print(f"Reload verification -- shape: {df_check.shape}, nulls: {df_check.isnull().sum().sum()}")
print(f"addr_state dtype: {df_check['addr_state'].dtype} (should be object/string)")
print(f"addr_state sample: {df_check['addr_state'].head().tolist()}")

# Verify temporal index
t_check = pd.read_parquet('data/temporal_split_index.parquet')
print(f"\nTemporal index -- shape: {t_check.shape}")
print(f"  Range: {t_check['issue_d'].min()} to {t_check['issue_d'].max()}")

# Check missingness indicators exist
miss_cols = [c for c in df_check.columns if c.endswith('_missing')]
print(f"\nMissingness indicators: {miss_cols}")

del df_check, t_check